# MOSFiT output visualization

Load products from a MOSFiT run and plot the photometric light curve, bolometric luminosity (if present), MCMC chain (if saved with `-c`), and a corner plot.

Requires `mosfit`, `corner`, `matplotlib`, `seaborn`, **`plotly`**, and Jupyter:

```bash
pip install plotly
```

Run this notebook from the `jupyter/` directory next to your run's `products/` folder (or set `MOSFIT_WALKERS` to a walkers file).


In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import logging
import os
from collections import OrderedDict, defaultdict
from pathlib import Path

import corner
from matplotlib.colors import to_hex

try:
    import plotly.graph_objects as go
except ImportError as exc:
    raise ImportError(
        'The interactive light-curve cell needs plotly. Install with:\n'
        '  pip install plotly'
    ) from exc
import h5py
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from tqdm.auto import tqdm

from mosfit.plotting import bandcolorf
from mosfit.utils import load_walkers_file

sns.reset_orig()
plt.rcParams['font.family'] = 'serif'
plt.rcParams.update({'font.size': 14})



def find_walkers_path():
    env = os.environ.get('MOSFIT_WALKERS', '').strip()
    if env:
        path = Path(env).expanduser()
        if path.is_file():
            return path
    for rel in ('../products/walkers.h5', 'products/walkers.h5',
                '../products/walkers.json', 'products/walkers.json'):
        path = Path(rel)
        if path.is_file():
            return path
    raise FileNotFoundError(
        'walkers.h5 not found. Run with notebook cwd=jupyter/, or export '
        'MOSFIT_WALKERS=/path/to/walkers.h5')




LSUN_ERG_S = 3.828e33
MBOL_SUN_MAG = 4.74


def L_to_absolute_mbol(L):
    """Absolute bolometric magnitude (Mbol,Sun = 4.74, Lsun = 3.828e33 erg/s)."""
    L = np.asarray(L, dtype=float)
    ratio = np.clip(L / LSUN_ERG_S, 1e-330, np.inf)
    return MBOL_SUN_MAG - 2.5 * np.log10(ratio)


def distance_modulus_mag(data, model):
    """Distance modulus μ from catalog or model lumdist (Mpc)."""
    lumdist = None
    if data.get('lumdist'):
        try:
            lumdist = float(data['lumdist'][0]['value'])
        except (KeyError, IndexError, TypeError, ValueError):
            lumdist = None
    if lumdist is None:
        for rz in model.get('realizations', []):
            p = rz.get('parameters', {}).get('lumdist')
            if p and p.get('value') is not None:
                lumdist = float(p['value'])
                break
    if lumdist is None or lumdist <= 0:
        return None, None
    return lumdist, 5.0 * np.log10(lumdist) + 25.0


walker_path = find_walkers_path()
data = load_walkers_file(str(walker_path))
if 'name' not in data:
    data = data[list(data.keys())[0]]

photo = data['photometry']
model = data['models'][0]

real_data = any(
    'band' in x and 'magnitude' in x and (
        'realization' not in x or 'simulated' in x)
    for x in photo)

band_attr = ['band', 'instrument', 'telescope', 'system', 'bandset']
band_list = list({
    tuple(x.get(y, '') for y in band_attr)
    for x in photo if 'band' in x and 'magnitude' in x})
real_band_list = list({
    tuple(x.get(y, '') for y in band_attr)
    for x in photo
    if 'band' in x and 'magnitude' in x and (
        'realization' not in x or 'simulated' in x)})

lum_catalog = [
    x for x in photo
    if 'luminosity' in x and x.get('realization') is None]
lum_model = [
    x for x in photo
    if 'luminosity' in x and x.get('realization') is not None]
has_luminosity_lc = bool(lum_catalog or lum_model)

print(f'Loaded {walker_path}')

lumdist_mpc, mu_mag = distance_modulus_mag(data, model)
if lumdist_mpc is not None:
    print(f'lumdist = {lumdist_mpc:g} Mpc  →  μ = {mu_mag:.3f} mag')
else:
    print('No lumdist found; absolute-magnitude axis will be omitted')


## Photometric light curve

Interactive Plotly figure (no Jupyter widgets/`anywidget` required): zoom/pan, hover catalog points for instrument and other metadata, apparent magnitude (left) and absolute magnitude (right) using catalog `lumdist`. Optionally restrict instruments with `inst_exclusive_list`.

**Deps:** `pip install plotly`.


In [ ]:
# Uncomment to plot only the listed instruments:
# inst_exclusive_list = ['UVOT']

META_KEYS = [
    'band', 'instrument', 'telescope', 'system', 'bandset',
    'source', 'u_time', 'upperlimit', 'e_magnitude',
    'e_lower_magnitude', 'e_upper_magnitude',
]


def _meta_text(ph):
    lines = []
    for k in META_KEYS:
        if k in ph and ph[k] not in (None, ''):
            lines.append(f'{k}: {ph[k]}')
    for k, v in sorted(ph.items()):
        if k in META_KEYS or k in ('time', 'magnitude', 'realization', 'simulated'):
            continue
        if isinstance(v, (str, int, float, bool)) and v != '':
            lines.append(f'{k}: {v}')
    return '<br>'.join(lines)


# Plain go.Figure (not FigureWidget) — works in JupyterLab without anywidget/ipywidgets.
fig = go.Figure()
labeled_bands = set()
ys_all = []

for full_band in tqdm(band_list, desc='Photo', leave=False):
    (band, inst, tele, syst, bset) = full_band
    if 'inst_exclusive_list' in globals() and inst not in inst_exclusive_list:
        continue
    color = to_hex(bandcolorf(band))

    realizations = [[] for _ in range(len(model['realizations']))]
    for ph in photo:
        rn = ph.get('realization', None)
        si = ph.get('simulated', False)
        if rn and not si:
            if tuple(ph.get(y, '') for y in band_attr) == full_band:
                realizations[int(rn) - 1].append((
                    float(ph['time']), float(ph['magnitude'])))
    for rz in realizations:
        if len(rz) < 2:
            continue
        xs, ys = zip(*sorted(rz, key=lambda p: p[0]))
        ys_all.extend(ys)
        fig.add_trace(go.Scatter(
            x=xs, y=ys, mode='lines',
            line=dict(color=color, width=1),
            opacity=0.25,
            hoverinfo='skip',
            showlegend=False,
        ))

    if real_data:
        for s, symb in enumerate(('circle', 'triangle-down')):
            cond = bool(s)
            pts = [
                x for x in photo
                if 'magnitude' in x and (
                    'realization' not in x or 'simulated' in x) and
                'host' not in x and 'includeshost' not in x and
                x.get('upperlimit', False) == cond and
                tuple(x.get(y, '') for y in band_attr) == full_band
            ]
            if not pts:
                continue
            xs = [float(x['time']) for x in pts]
            ys = [float(x['magnitude']) for x in pts]
            ys_all.extend(ys)
            yerr = []
            for x in pts:
                if x.get('upperlimit'):
                    yerr.append(0.0)
                else:
                    yerr.append(float(
                        x.get('e_magnitude',
                              x.get('e_upper_magnitude',
                                    x.get('e_lower_magnitude', 0.0)))))
            show = band not in labeled_bands
            hover = (
                'MJD %{x:.3f}<br>apparent m = %{y:.3f}'
                + (('<br>absolute M ≈ %{customdata[0]:.3f}'
                    if mu_mag is not None else ''))
                + '<br>%{customdata[1]}<extra></extra>'
            )
            fig.add_trace(go.Scatter(
                x=xs, y=ys, mode='markers',
                name=band,
                legendgroup=band,
                showlegend=show,
                marker=dict(
                    color=color, size=9, symbol=symb,
                    line=dict(color='black', width=1)),
                error_y=dict(
                    type='data', array=yerr, visible=True,
                    color=color, thickness=1.2, width=3),
                hovertemplate=hover,
                customdata=[
                    [
                        (float(x['magnitude']) - mu_mag) if mu_mag is not None else None,
                        _meta_text(x),
                    ]
                    for x in pts
                ],
            ))
            if show:
                labeled_bands.add(band)

for full_band in band_list:
    band = full_band[0]
    if band in labeled_bands:
        continue
    if 'inst_exclusive_list' in globals() and full_band[1] not in inst_exclusive_list:
        continue
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='lines',
        name=band, line=dict(color=to_hex(bandcolorf(band)), width=2),
        hoverinfo='skip'))
    labeled_bands.add(band)

pad = 0.5
if ys_all:
    y0, y1 = min(ys_all), max(ys_all)
else:
    y0, y1 = 10.0, 20.0
ticks = np.arange(np.floor(y0 - pad), np.ceil(y1 + pad) + 0.5, 1.0)
y_range = [y1 + pad, y0 - pad]  # inverted

layout = dict(
    width=950, height=600,
    title=data.get('name', 'MOSFiT light curve'),
    xaxis_title='MJD',
    yaxis=dict(
        title='Apparent magnitude',
        range=list(y_range),
        tickvals=ticks,
        ticktext=[f'{t:.0f}' for t in ticks],
    ),
    legend_title_text='Filter',
    hovermode='closest',
    template='plotly_white',
    margin=dict(l=60, r=70, t=50, b=50),
)

if mu_mag is not None:
    # Same data coords as apparent (matches='y'); right ticks show m − μ.
    # Zoom stays synced without FigureWidget/anywidget.
    layout['yaxis2'] = dict(
        title=f'Absolute magnitude (μ = {mu_mag:.2f})',
        overlaying='y',
        side='right',
        range=list(y_range),
        matches='y',
        tickvals=ticks,
        ticktext=[f'{t - mu_mag:.1f}' for t in ticks],
        showgrid=False,
    )

fig.update_layout(**layout)
fig.show()

fig.write_html('../products/lc.html')
print('Wrote ../products/lc.html')


## Bolometric luminosity

Photometry rows with `luminosity` (erg s$^{-1}$). Catalog points omit `realization`; model curves include it.


In [ ]:
if not has_luminosity_lc:
    print('No luminosity rows in walkers file — skipping bolometric panels.')
else:
    fig, (ax_log, ax_mbol) = plt.subplots(
        2, 1, figsize=(12, 9), sharex=True,
        gridspec_kw={'height_ratios': [1.15, 1.0]}, constrained_layout=True)

    rows_cat = sorted(lum_catalog, key=lambda r: float(r['time']))
    if rows_cat:
        tx = np.array([float(r['time']) for r in rows_cat], dtype=float)
        Ll = np.array([float(r['luminosity']) for r in rows_cat], dtype=float)
        eL = np.full_like(Ll, np.nan)
        for i, r in enumerate(rows_cat):
            ev = r.get('e_luminosity')
            if ev is None:
                continue
            try:
                eL[i] = float(ev)
            except (TypeError, ValueError):
                pass

        mbar = np.isfinite(eL) & (eL > 0) & (Ll > 0)
        ax_log.errorbar(
            tx[mbar], Ll[mbar], yerr=eL[mbar], fmt='none',
            elinewidth=1.3, capsize=3.5, color='0.35', alpha=0.75, zorder=3)
        ax_log.scatter(
            tx, Ll, marker='s', s=105, lw=2, edgecolor='k',
            facecolor='w', label='catalog $L$', zorder=5)

        Mb = L_to_absolute_mbol(Ll)
        eMb = np.full_like(Ll, np.nan)
        ok_e = np.isfinite(eL) & (eL > 0) & (Ll > 0)
        eMb[ok_e] = (2.5 / np.log(10.0)) * (eL[ok_e] / Ll[ok_e])
        ax_mbol.errorbar(
            tx, Mb, yerr=eMb, fmt='none', elinewidth=1.25, capsize=3.,
            color='0.35', alpha=0.7)
        ax_mbol.scatter(
            tx, Mb, marker='s', s=95, lw=2, edgecolor='k',
            facecolor='w', label=r'catalog $M_{\rm bol}$', zorder=5)

    by_rn = defaultdict(list)
    for r in lum_model:
        by_rn[r['realization']].append(
            (float(r['time']), float(r['luminosity'])))

    for rn, pts in tqdm(
            sorted(by_rn.items(), key=lambda kv: kv[0]),
            desc='Lum walkers', leave=False):
        pts = sorted(pts, key=lambda q: q[0])
        if not pts:
            continue
        tplot, Ly = zip(*pts)
        Ly = np.asarray(Ly, dtype=float)
        ax_log.plot(tplot, Ly, color='steelblue', lw=1.6, alpha=0.075)
        ax_mbol.plot(
            tplot, L_to_absolute_mbol(Ly),
            color='steelblue', lw=1.5, alpha=0.065)

    if by_rn:
        longest = sorted(by_rn.values(), key=len)[-1]
        longest = sorted(longest, key=lambda p: p[0])
        tl, Ll2 = zip(*longest)
        Ll2 = np.asarray(Ll2, dtype=float)
        ax_log.plot(
            tl, Ll2, color='steelblue', lw=2.75, alpha=0.92,
            label='one realization (thin = walkers)')
        ax_mbol.plot(
            tl, L_to_absolute_mbol(Ll2), color='steelblue', lw=2.5,
            alpha=0.92)

    ax_log.set_yscale('log')
    ax_log.set_ylabel(
        r'$L_{\mathrm{bol}}~(\mathrm{erg}\,\mathrm{s}^{-1})$')
    ax_log.legend(loc='best', fontsize=11, framealpha=0.95)

    ax_mbol.invert_yaxis()
    ax_mbol.set_xlabel('Epoch (times as stored in walkers)')
    ax_mbol.set_ylabel(r'Absolute bolometric magnitude $M_{\rm bol}$')
    plt.show()


## Chain evolution

If the fit was run with `-c`, MOSFiT writes `products/chain.h5`. This cell plots walker traces for each free parameter.

In [ ]:
chain_candidates = [
    Path('../products/chain.h5'), Path('products/chain.h5')]
chain_path = next((p for p in chain_candidates if p.is_file()), None)

if chain_path is None:
    print('No chain.h5 found — re-run with -c to enable this plot.')
else:
    with h5py.File(chain_path, 'r') as hf:
        all_chain = hf['samples'][:]
        param_names = [
            n.decode() if isinstance(n, bytes) else n
            for n in hf['param_names'][:]]

    nparam = all_chain.shape[-1]
    fig = plt.figure(figsize=(4. * np.ceil(nparam / 4.), 8))
    for pi in range(nparam):
        my_chain = all_chain[0, :, :, pi]
        ax = fig.add_subplot(int(np.ceil(nparam / 4.)), 4, pi + 1)
        ax.plot(my_chain.T)
        ax.plot(np.mean(my_chain, axis=0), color='k')
        ax.set_title(param_names[pi])
    plt.tight_layout()
    plt.show()
    print(f'Loaded {chain_path}')

## Corner plot

Posterior corner plot from walker realizations (requires the [`corner`](https://corner.readthedocs.io) package).

In [ ]:
logging.disable(logging.WARNING)

corner_input = []
pars = [
    x for x in model['setup']
    if model['setup'][x].get('kind') == 'parameter' and
    'min_value' in model['setup'][x] and 'max_value' in model['setup'][x]]
weights = []
var_names = None
for realization in model['realizations']:
    par_vals = realization['parameters']
    if 'weight' in realization:
        weights.append(float(realization['weight']))
    var_names = [
        '$' + ('\\log\\, ' if par_vals[x].get('log') else '') +
        par_vals[x]['latex'] + '$'
        for x in par_vals if x in pars and 'fraction' in par_vals[x]]
    corner_input.append([
        np.log10(par_vals[x]['value']) if par_vals[x].get('log')
        else par_vals[x]['value']
        for x in par_vals if x in pars and 'fraction' in par_vals[x]])

# Newer corner/arviz need an ndarray, not a list of lists.
corner_input = np.asarray(corner_input, dtype=float)
weights = np.asarray(weights, dtype=float) if weights else None
ranges = [0.999] * corner_input.shape[1]
cfig = corner.corner(
    corner_input, labels=var_names, quantiles=[0.16, 0.5, 0.84],
    show_titles=True, weights=weights, range=ranges)
cfig.savefig('../products/corner.pdf')


These cells are a starting point for inspecting MOSFiT products. Adapt them for your own analysis notebooks.